# R31 — Two-Stage Cascade for Referable Glaucoma

**Task:** Detect **referable (bilateral) glaucoma** — patients with ≥ 2 images showing `increased_cup_disc = 1`.

## Approach
1. **Stage 1 (per-eye):** Retrain R28's per-eye `increased_cup_disc` MLPs using saved optimal hyperparameters (skip BayesSearchCV).
2. **Stage 2 (bilateral counting):** Threshold per-image probabilities → count positive eyes per patient → referable if ≥ 2.

Threshold is swept on the **val set** to maximize **patient-level referable F1**, then applied to **test set**.

Each of the 7 encoders is evaluated independently (same structure as R28).

In [13]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1 — Imports
# ═══════════════════════════════════════════════════════════════════════
from __future__ import annotations
import json, time, warnings, ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.collections import LineCollection

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, balanced_accuracy_score,
    confusion_matrix, classification_report, roc_curve,
)

import sys
sys.path.insert(0, str(Path.cwd() / 'src'))
from retina_embeddings_dataset import load_retina_embeddings_dataset

warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('seaborn-v0_8-whitegrid')
print('R31 imports OK')

R31 imports OK


In [14]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 2 — Configuration
# ═══════════════════════════════════════════════════════════════════════

# ─── Task ─────────────────────────────────────────────────────────
DATASET_NAME = 'brset'
TASK         = 'increased_cup_disc'   # per-eye label used by Stage 1
RANDOM_SEED  = 42

from multiprocessing import cpu_count
CPU_COUNT = cpu_count()

# ─── Paths ────────────────────────────────────────────────────────
BASE_DIR     = Path.cwd()
FE_DIR       = BASE_DIR / 'artifacts' / 'brset_eda_fe'
EMBED_DIR    = BASE_DIR / 'data' / 'brset_embeddings'
LABELS_PATH  = EMBED_DIR / 'brset_labels' / 'labels_brset.csv'
RESULTS_DIR  = BASE_DIR / 'results' / 'brset_r31_cascade_referable_glaucoma'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# R28 saved hyperparameters
R28_DIR = BASE_DIR / 'results' / 'brset_r28_fe_experiment' / 'r28_per_encoder'

# ─── Pipeline parameters (identical to R28) ───────────────────────
PCA_COMPONENTS       = 384
MI_SELECTION         = True
MI_PERCENTILE_THRESH = 0.15
MI_MIN_COMPONENTS    = 128

MLP_MAX_ITER   = 400
MLP_PATIENCE   = 30

BAG_SEEDS           = [42, 123, 314, 555, 789]
N_BAG_ARCH_VARIANTS = 5
N_BAGS = len(BAG_SEEDS) * N_BAG_ARCH_VARIANTS   # 25

THRESHOLD_GRID  = 301
THRESHOLD_FLOOR = 0.05

# ─── Clinical metadata (same 5 cols as R28) ──────────────────────
METADATA_COLS = {
    'age':      'numeric',
    'sex':      'passthrough',
    'diabetes': 'binary_yn',
    'exam_eye': 'binary',
    'camera':   'camera',
}

# ─── optic_disc masking (same as R28) ────────────────────────────
OD_P_MASK_BASE    = 0.50
OD_P_MASK_RANGE   = (0.35, 0.70)
OD_CORRUPT        = 0.10
OD_FEATURE_WEIGHT = 0.50
OD_MISS_WEIGHT    = 0.50
OD_TEST_P_MASK    = 0.25

# ─── Bilateral / referable threshold ─────────────────────────────
BILATERAL_MIN_EYES = 2   # ≥ 2 positive images → referable

print(f'R28 hyper dir:   {R28_DIR}')
print(f'Results dir:     {RESULTS_DIR}')
print(f'Stage 1 task:    {TASK} (per-eye)')
print(f'Stage 2 rule:    ≥{BILATERAL_MIN_EYES} positive eyes → referable')
print(f'PCA: {PCA_COMPONENTS} | MI: drop <{MI_PERCENTILE_THRESH:.0%} | min {MI_MIN_COMPONENTS}')
print(f'Bagging: {len(BAG_SEEDS)} seeds × {N_BAG_ARCH_VARIANTS} variants = {N_BAGS}')
print(f'OD masking: base={OD_P_MASK_BASE:.0%}, range={OD_P_MASK_RANGE}, corrupt={OD_CORRUPT:.0%}')

R28 hyper dir:   c:\Users\Julian\OneDrive\Desktop\testing-retina-project\Retina-Project-Evaluation\main-project\results\brset_r28_fe_experiment\r28_per_encoder
Results dir:     c:\Users\Julian\OneDrive\Desktop\testing-retina-project\Retina-Project-Evaluation\main-project\results\brset_r31_cascade_referable_glaucoma
Stage 1 task:    increased_cup_disc (per-eye)
Stage 2 rule:    ≥2 positive eyes → referable
PCA: 384 | MI: drop <15% | min 128
Bagging: 5 seeds × 5 variants = 25
OD masking: base=50%, range=(0.35, 0.7), corrupt=10%


In [15]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 3 — Load Data + FE Artifacts
# ═══════════════════════════════════════════════════════════════════════

# ─── 3-way patient split ──────────────────────────────────────────
splits = pd.read_csv(FE_DIR / 'splits_patient.csv')
splits['patient_id'] = splits['patient_id'].astype(str)

train_pids = set(splits[splits['split'] == 'train']['patient_id'])
val_pids   = set(splits[splits['split'] == 'val']['patient_id'])
test_pids  = set(splits[splits['split'] == 'test']['patient_id'])

assert len(train_pids & val_pids)  == 0, 'Train/val patient leakage!'
assert len(train_pids & test_pids) == 0, 'Train/test patient leakage!'
assert len(val_pids   & test_pids) == 0, 'Val/test patient leakage!'

# ─── Labels for clinical metadata ─────────────────────────────────
labels_raw = pd.read_csv(LABELS_PATH)
labels_raw['patient_id'] = labels_raw['patient_id'].astype(str)
labels_raw = labels_raw.rename(columns={'patient_age': 'age', 'patient_sex': 'sex'})

# ─── Embedding files ──────────────────────────────────────────────
embedding_files = sorted(EMBED_DIR.glob('Embeddings_*.csv'))

# ─── Build patient-level ground truth for referable glaucoma ──────
#     A patient is "referable" if they have ≥ BILATERAL_MIN_EYES images
#     with increased_cup_disc == 1
def build_referable_labels(df, task_col, min_eyes):
    """Return Series: patient_id → 1 (referable) or 0."""
    pos_count = (
        df[df[task_col] == 1]
        .groupby('patient_id')[task_col]
        .count()
    )
    all_pids = df['patient_id'].unique()
    result = pd.Series(0, index=all_pids, name='referable_glaucoma')
    referable = pos_count[pos_count >= min_eyes].index
    result.loc[result.index.isin(referable)] = 1
    return result

# Preview patient distribution
_preview_df = labels_raw.dropna(subset=[TASK]).copy()
_preview_df['patient_id'] = _preview_df['patient_id'].astype(str)
_ref = build_referable_labels(_preview_df, TASK, BILATERAL_MIN_EYES)

_ref_train = _ref[_ref.index.isin(train_pids)]
_ref_val   = _ref[_ref.index.isin(val_pids)]
_ref_test  = _ref[_ref.index.isin(test_pids)]

print(f'Patient split:   {len(train_pids)} train / {len(val_pids)} val / {len(test_pids)} test')
print(f'Embedding files: {len(embedding_files)}')
print(f'\nReferable glaucoma (≥{BILATERAL_MIN_EYES} eyes with {TASK}=1):')
print(f'  Train: {_ref_train.sum():>4}/{len(_ref_train)} ({_ref_train.mean():.1%})')
print(f'  Val:   {_ref_val.sum():>4}/{len(_ref_val)} ({_ref_val.mean():.1%})')
print(f'  Test:  {_ref_test.sum():>4}/{len(_ref_test)} ({_ref_test.mean():.1%})')

# ─── Load R28 hyperparameters for all encoders ────────────────────
r28_hypers = {}
for d in sorted(R28_DIR.iterdir()):
    hp_file = d / 'hyperparameters.json'
    if hp_file.exists():
        with open(hp_file) as f:
            hp = json.load(f)
        r28_hypers[hp['embedding']] = hp
        print(f'  R28 hypers loaded: {hp["embedding"]}  '
              f'arch={hp["architecture"]}  α={hp["alpha"]:.4f}  '
              f'lr={hp["lr"]:.6f}  w={hp["pos_weight"]:.2f}  '
              f'agg={hp["best_agg"]}')

print(f'\nLoaded R28 hyperparameters for {len(r28_hypers)} encoders')

Patient split:   5796 train / 1023 val / 1705 test
Embedding files: 7

Referable glaucoma (≥2 eyes with increased_cup_disc=1):
  Train:  789/5796 (13.6%)
  Val:    157/1023 (15.3%)
  Test:   229/1705 (13.4%)
  R28 hypers loaded: convnextv2_base_  arch=(128,)  α=1.0000  lr=0.000100  w=4.07  agg=max
  R28 hypers loaded: dinov3_convnext_base  arch=(128,)  α=1.0000  lr=0.000358  w=2.46  agg=mean
  R28 hypers loaded: dinov3_vitb16  arch=(128,)  α=1.0000  lr=0.000301  w=2.45  agg=noisy_or
  R28 hypers loaded: RETFound_dinov2_shanghai  arch=(512,)  α=1.0000  lr=0.000100  w=2.91  agg=max
  R28 hypers loaded: RETFound_mae_natureCFP  arch=(512,)  α=0.0010  lr=0.000210  w=3.37  agg=max
  R28 hypers loaded: RETFound_mae_shanghai  arch=(256,)  α=0.0014  lr=0.002000  w=1.00  agg=noisy_or
  R28 hypers loaded: vit_base_  arch=(256,)  α=0.0010  lr=0.002000  w=2.62  agg=max

Loaded R28 hyperparameters for 7 encoders


In [16]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 4 — Helper Functions
# ═══════════════════════════════════════════════════════════════════════

def short_name(path):
    stem = Path(path).stem
    for s in ['Embeddings_brset_', 'Embeddings_']:
        stem = stem.replace(s, '')
    return stem


def make_sample_weights(y, pos_class_weight=1.0):
    y = np.asarray(y)
    w = np.where(y == 1, float(pos_class_weight), 1.0).astype(np.float32)
    mean_w = w.mean()
    if mean_w > 0:
        w /= mean_w
    return w


def mi_select_features(X_train, y_train, X_others,
                       percentile_thresh=0.15, min_components=128,
                       random_state=42):
    mi_scores = mutual_info_classif(
        X_train, y_train, discrete_features=False,
        n_neighbors=5, random_state=random_state,
    )
    threshold = np.percentile(mi_scores, percentile_thresh * 100)
    mask = mi_scores >= threshold
    if mask.sum() < min_components:
        top_k = np.argsort(mi_scores)[-min_components:]
        mask = np.zeros(len(mi_scores), dtype=bool)
        mask[top_k] = True
    X_train_sel = X_train[:, mask]
    X_others_sel = [X[:, mask] for X in X_others]
    return X_train_sel, X_others_sel, mask, mi_scores


# ─── Clinical metadata encoding (same as R28) ─────────────────────
def encode_metadata(df, metadata_cols, fit_stats=None):
    if fit_stats is None:
        fit_stats = {}
    meta_parts = []
    for col, dtype in metadata_cols.items():
        if col not in df.columns:
            print(f'    WARNING: metadata col {col!r} not found, skipping')
            continue
        vals = df[col].copy()
        if dtype == 'numeric':
            vals = pd.to_numeric(vals, errors='coerce')
            if 'median_' + col not in fit_stats:
                fit_stats['median_' + col] = float(vals.median())
            vals = vals.fillna(fit_stats['median_' + col])
            meta_parts.append(vals.to_numpy(dtype=np.float32).reshape(-1, 1))
        elif dtype == 'binary':
            vals = pd.to_numeric(vals, errors='coerce').fillna(1)
            vals = (vals - 1).clip(0, 1)
            meta_parts.append(vals.to_numpy(dtype=np.float32).reshape(-1, 1))
        elif dtype == 'passthrough':
            vals = pd.to_numeric(vals, errors='coerce').fillna(0)
            meta_parts.append(vals.to_numpy(dtype=np.float32).reshape(-1, 1))
        elif dtype == 'binary_yn':
            vals = vals.astype(str).str.lower().map({'yes': 1.0, 'no': 0.0}).fillna(0.0)
            meta_parts.append(vals.to_numpy(dtype=np.float32).reshape(-1, 1))
        elif dtype == 'camera':
            vals = vals.astype(str).map({'Canon CR': 0.0, 'NIKON NF5050': 1.0}).fillna(0.0)
            meta_parts.append(vals.to_numpy(dtype=np.float32).reshape(-1, 1))
    if not meta_parts:
        return np.zeros((len(df), 0), dtype=np.float32), fit_stats
    return np.hstack(meta_parts), fit_stats


# ─── Architecture variant helpers (same as R28) ──────────────────
def make_wider(arch, factor=1.5):
    return tuple(max(64, int(x * factor)) for x in arch)

def make_deeper(arch):
    return arch + (max(64, arch[-1] // 2),)

def make_narrower(arch, factor=0.67):
    return tuple(max(64, int(x * factor)) for x in arch)

def make_shallower(arch):
    if len(arch) > 1:
        return arch[:-1]
    return (max(64, arch[0] // 2),)


# ─── optic_disc masking (same as R28) ─────────────────────────────
def _bin_od(series):
    return (pd.to_numeric(series, errors='coerce') == 2).astype(np.float32).to_numpy()

def _mask_od(od_series, p_mask, rng, p_corrupt=0.0):
    od = _bin_od(od_series)
    if p_mask >= 1.0:
        miss = np.ones(len(od), dtype=bool)
    elif p_mask > 0:
        miss = rng.random(len(od)) < p_mask
    else:
        miss = np.zeros(len(od), dtype=bool)
    od_m = od.copy()
    od_m[miss] = 0.0
    if p_corrupt and p_corrupt > 0:
        corrupt = (~miss) & (rng.random(len(od)) < p_corrupt)
        od_m[corrupt] = 1.0 - od_m[corrupt]
    return od_m.reshape(-1, 1), miss.astype(np.float32).reshape(-1, 1)

def _apply_od_weights(X_s, od_w, miss_w):
    if X_s.shape[1] < 2:
        return X_s
    if od_w != 1.0:
        X_s[:, -2] *= od_w
    if miss_w != 1.0:
        X_s[:, -1] *= miss_w
    return X_s


# ─── Bilateral cascade logic (NEW for R31) ────────────────────────
def bilateral_patient_predictions(pids, proba, threshold, min_eyes=2):
    """
    Stage 2: For each patient, count images with proba >= threshold.
    Patient is referable if count >= min_eyes.
    Returns: patient_id → (referable_pred: 0/1, max_proba, n_positive_eyes)
    """
    df = pd.DataFrame({
        'patient_id': pids,
        'proba': proba,
        'pred_pos': (proba >= threshold).astype(int),
    })
    agg = df.groupby('patient_id').agg(
        n_positive=('pred_pos', 'sum'),
        n_images=('pred_pos', 'count'),
        max_proba=('proba', 'max'),
        mean_proba=('proba', 'mean'),
    )
    agg['referable_pred'] = (agg['n_positive'] >= min_eyes).astype(int)
    return agg


def sweep_bilateral_threshold(pids, proba, y_referable_patient,
                              min_eyes=2, grid_size=301, floor=0.05):
    """
    Sweep per-image thresholds on val set.
    y_referable_patient: Series (patient_id → 0/1 referable label).
    Returns: best_threshold, best_f1, all_results
    """
    thresholds = np.linspace(floor, 0.99, grid_size)
    best_f1, best_t = 0.0, 0.5
    all_res = []

    for t in thresholds:
        agg = bilateral_patient_predictions(pids, proba, t, min_eyes)
        com = agg.index.intersection(y_referable_patient.index)
        if len(com) == 0:
            continue
        yt = y_referable_patient.loc[com].values
        yh = agg.loc[com, 'referable_pred'].values
        if yh.sum() == 0:
            f1 = 0.0
        else:
            f1 = f1_score(yt, yh, zero_division=0)
        prec = precision_score(yt, yh, zero_division=0)
        rec  = recall_score(yt, yh, zero_division=0)
        all_res.append({'threshold': t, 'f1': f1, 'precision': prec, 'recall': rec,
                        'n_predicted_pos': int(yh.sum())})
        if f1 > best_f1:
            best_f1, best_t = f1, t

    return best_t, best_f1, pd.DataFrame(all_res)


# ─── Plotting (same as R28) ───────────────────────────────────────
def plot_confusion_matrix(cm, title, save_path=None, show=True):
    cm = np.asarray(cm)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    cm_plot = np.ma.masked_where(cm <= 0, cm)
    vmax = int(cm.max()) if cm.size else 1
    norm = LogNorm(vmin=1, vmax=max(1, vmax))
    im = ax.imshow(cm_plot, cmap='viridis', interpolation='nearest', norm=norm)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label('Count (log)', fontsize=9)
    ax.set_title(title, fontsize=11, pad=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticks(range(cm.shape[1])); ax.set_yticks(range(cm.shape[0]))
    thresh = cm.max() / 2.0
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, f'{val}', ha='center', va='center',
                color='white' if val > thresh else 'black', fontsize=10, fontweight='bold')
    plt.tight_layout()
    if save_path: fig.savefig(save_path, dpi=160, bbox_inches='tight')
    if show: plt.show()
    else: plt.close(fig)


def plot_roc_curve(fpr, tpr, auc_val, title, save_path=None, show=True):
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    points = np.array([fpr, tpr]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = LineCollection(segments, cmap='plasma', norm=Normalize(0, 1))
    lc.set_array(fpr); lc.set_linewidth(2.6)
    ax.add_collection(lc)
    ax.plot([0,1],[0,1],'--', color='#666', lw=1.2, label='Chance')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_title(f'{title}\nAUC = {auc_val:.3f}', fontsize=11, pad=12)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(loc='lower right', fontsize=9)
    fig.colorbar(lc, ax=ax, fraction=0.046, pad=0.04).set_label('FPR', fontsize=9)
    plt.tight_layout()
    if save_path: fig.savefig(save_path, dpi=160, bbox_inches='tight')
    if show: plt.show()
    else: plt.close(fig)


print(f'Helpers OK  (bilateral min eyes = {BILATERAL_MIN_EYES})')

Helpers OK  (bilateral min eyes = 2)


---
## Per-Encoder Two-Stage Cascade

For each encoder:
1. **Load R28 hyperparameters** (skip BayesSearchCV entirely)
2. **Retrain 25 bagged MLPs** on `increased_cup_disc` with R28's optimal params
3. **Get per-image probabilities** (calibrated with IsotonicRegression)
4. **Bilateral threshold sweep** on val: threshold per-image proba → count positive eyes → referable if ≥ 2 → maximize F1
5. **Evaluate on test** with best bilateral threshold → patient-level metrics

In [17]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 5 — Per-Encoder Cascade Loop
# ═══════════════════════════════════════════════════════════════════════

R31_DIR = RESULTS_DIR / 'r31_per_encoder'
R31_DIR.mkdir(parents=True, exist_ok=True)

print('═' * 80)
print('R31 — Two-Stage Cascade: Per-Eye MLP → Bilateral Counting')
print('═' * 80)

r31_results = []
t0_global = time.time()

for emb_idx, emb_path in enumerate(embedding_files):
    sname = short_name(emb_path)

    # ── Checkpoint: skip if results already exist ──────────────────
    chk_dir = R31_DIR / f'{emb_idx:02d}_{sname}'
    chk_hp  = chk_dir / 'hyperparameters.json'
    if chk_hp.exists():
        print(f'\n[{emb_idx+1}/{len(embedding_files)}] {sname} — CACHED, loading')
        with open(chk_hp) as _f:
            prev = json.load(_f)
        r31_results.append({
            'embedding': sname,
            'dim_raw': prev.get('emb_dim_raw', 0),
            'dim_pca': prev['pca_dim'],
            'dim_mi': prev['mi_selected'],
            'pca_var': prev['pca_var'],
            'r28_arch': prev['r28_architecture'],
            'r28_alpha': prev['r28_alpha'],
            'r28_lr': prev['r28_lr'],
            'r28_pos_weight': prev['r28_pos_weight'],
            'bilateral_threshold': prev['bilateral_threshold'],
            'val_referable_f1': prev['val_referable_f1'],
            'test_referable_f1': prev['test_referable_f1'],
            'test_referable_precision': prev['test_referable_precision'],
            'test_referable_recall': prev['test_referable_recall'],
            'test_referable_auc': prev['test_referable_auc'],
            'test_referable_bal_acc': prev['test_referable_bal_acc'],
            'n_test_patients': prev['n_test_patients'],
            'time_s': 0,
        })
        continue

    # ── Check R28 hyperparameters exist for this encoder ──────────
    if sname not in r28_hypers:
        print(f'\n[{emb_idx+1}/{len(embedding_files)}] {sname} — SKIPPED, no R28 hypers found')
        continue

    hp = r28_hypers[sname]
    barch  = ast.literal_eval(hp['architecture']) if isinstance(hp['architecture'], str) else tuple(hp['architecture'])
    balpha = hp['alpha']
    blr    = hp['lr']
    bbs    = hp['batch_size']
    bw     = hp['pos_weight']

    print(f'\n{"═"*80}')
    print(f'[{emb_idx+1}/{len(embedding_files)}] R31 — {sname}')
    print(f'  R28 hypers: arch={barch} α={balpha:.4f} lr={blr:.6f} bs={bbs} w={bw:.2f}')
    print(f'{"═"*80}')
    t0 = time.time()

    # ══════════════════════════════════════════════════════════════
    # STAGE 1: Train per-eye increased_cup_disc MLPs (using R28 params)
    # ══════════════════════════════════════════════════════════════

    # ── Load & split ──────────────────────────────────────────────
    ds = load_retina_embeddings_dataset(
        dataset=DATASET_NAME, embeddings_csv_path=emb_path,
        labels_csv_path=LABELS_PATH,
    )
    df = ds.df.copy()
    df['patient_id'] = df['patient_id'].astype(str)

    b_train = df[df['patient_id'].isin(train_pids)].dropna(subset=[TASK]).copy()
    b_val   = df[df['patient_id'].isin(val_pids)].dropna(subset=[TASK]).copy()
    b_test  = df[df['patient_id'].isin(test_pids)].dropna(subset=[TASK]).copy()

    X_tr_emb = b_train[ds.feature_cols].to_numpy(dtype=np.float32)
    X_va_emb = b_val[ds.feature_cols].to_numpy(dtype=np.float32)
    X_te_emb = b_test[ds.feature_cols].to_numpy(dtype=np.float32)

    yb_train = b_train[TASK].to_numpy(dtype=int)
    yb_val   = b_val[TASK].to_numpy(dtype=int)
    yb_test  = b_test[TASK].to_numpy(dtype=int)

    pids_tr = b_train['patient_id'].to_numpy()
    pids_va = b_val['patient_id'].to_numpy()
    pids_te = b_test['patient_id'].to_numpy()

    emb_dim_raw = X_tr_emb.shape[1]

    # ── PCA ───────────────────────────────────────────────────────
    nc = min(PCA_COMPONENTS, emb_dim_raw, X_tr_emb.shape[0])
    pca = PCA(n_components=nc, random_state=RANDOM_SEED)
    X_tr_pca = pca.fit_transform(X_tr_emb)
    X_va_pca = pca.transform(X_va_emb)
    X_te_pca = pca.transform(X_te_emb)
    var_exp = pca.explained_variance_ratio_.sum()
    print(f'  PCA: {emb_dim_raw}→{nc} ({var_exp:.1%} var)')

    # ── MI Selection ──────────────────────────────────────────────
    if MI_SELECTION:
        X_tr_sel, (X_va_sel, X_te_sel), mi_mask, _ = mi_select_features(
            X_tr_pca, yb_train, [X_va_pca, X_te_pca],
            percentile_thresh=MI_PERCENTILE_THRESH,
            min_components=MI_MIN_COMPONENTS,
            random_state=RANDOM_SEED,
        )
        n_kept = mi_mask.sum()
        print(f'  MI: {nc}→{n_kept}')
    else:
        X_tr_sel, X_va_sel, X_te_sel = X_tr_pca, X_va_pca, X_te_pca
        n_kept = nc

    # ── Metadata ──────────────────────────────────────────────────
    X_tr_meta, ms = encode_metadata(b_train, METADATA_COLS)
    X_va_meta, _  = encode_metadata(b_val, METADATA_COLS, fit_stats=ms)
    X_te_meta, _  = encode_metadata(b_test, METADATA_COLS, fit_stats=ms)
    meta_dim = X_tr_meta.shape[1]

    # ── OD masking for training ───────────────────────────────────
    rng_hp = np.random.default_rng(RANDOM_SEED)
    od_tr, miss_tr = _mask_od(b_train['optic_disc'], OD_P_MASK_BASE, rng_hp, OD_CORRUPT)
    X_tr_full = np.hstack([X_tr_sel, X_tr_meta, od_tr, miss_tr])
    ndim = X_tr_full.shape[1]
    print(f'  Dims: {n_kept} PCA + {meta_dim} meta + 2 OD = {ndim}')

    # ── Scale ─────────────────────────────────────────────────────
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr_full)
    X_tr_s = _apply_od_weights(X_tr_s, OD_FEATURE_WEIGHT, OD_MISS_WEIGHT)

    # ── Pre-compute val/test with P25 OD masking ──────────────────
    rng_v = np.random.default_rng(RANDOM_SEED + 7000)
    rng_t = np.random.default_rng(RANDOM_SEED + 9999)
    od_v, miss_v = _mask_od(b_val['optic_disc'],  OD_TEST_P_MASK, rng_v)
    od_t, miss_t = _mask_od(b_test['optic_disc'], OD_TEST_P_MASK, rng_t)

    X_va_s = scaler.transform(np.hstack([X_va_sel, X_va_meta, od_v, miss_v]))
    X_te_s = scaler.transform(np.hstack([X_te_sel, X_te_meta, od_t, miss_t]))
    X_va_s = _apply_od_weights(X_va_s.copy(), OD_FEATURE_WEIGHT, OD_MISS_WEIGHT)
    X_te_s = _apply_od_weights(X_te_s.copy(), OD_FEATURE_WEIGHT, OD_MISS_WEIGHT)

    # ── Bagging (25 MLPs, NO BayesSearchCV) ───────────────────────
    barchs = [barch, make_wider(barch), make_deeper(barch),
              make_narrower(barch), make_shallower(barch)]
    print(f'  [Bagging] {N_BAGS} models (R28 hypers, no search)')

    cal_p = {}
    cal_l = {}
    bag_va_proba = []
    bag_te_proba = []

    for s_idx, seed in enumerate(BAG_SEEDS):
        upids = np.unique(pids_tr)
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
        plm = b_train.groupby('patient_id')[TASK].max().to_dict()
        pl  = np.array([plm[p] for p in upids])
        ti, vi = next(sss.split(upids, pl))
        itr_pids = set(upids[ti])
        iva_pids = set(upids[vi])

        tr_m = np.isin(pids_tr, list(itr_pids))
        va_m = np.isin(pids_tr, list(iva_pids))

        rng_pm = np.random.default_rng(seed + 90000)
        pm_seed = float(rng_pm.uniform(OD_P_MASK_RANGE[0], OD_P_MASK_RANGE[1]))

        # Inner train
        rng_it = np.random.default_rng(seed + 10000)
        od_it, miss_it = _mask_od(
            b_train.iloc[np.where(tr_m)[0]]['optic_disc'], pm_seed, rng_it, OD_CORRUPT)
        X_it = scaler.transform(np.hstack([X_tr_sel[tr_m], X_tr_meta[tr_m], od_it, miss_it]))
        X_it = _apply_od_weights(X_it, OD_FEATURE_WEIGHT, OD_MISS_WEIGHT)
        y_it = yb_train[tr_m]

        # Inner val (for calibration)
        rng_iv = np.random.default_rng(seed + 20000)
        od_iv, miss_iv = _mask_od(
            b_train.iloc[np.where(va_m)[0]]['optic_disc'], pm_seed, rng_iv, OD_CORRUPT)
        X_iv = scaler.transform(np.hstack([X_tr_sel[va_m], X_tr_meta[va_m], od_iv, miss_iv]))
        X_iv = _apply_od_weights(X_iv, OD_FEATURE_WEIGHT, OD_MISS_WEIGHT)
        y_iv = yb_train[va_m]

        svp = []
        for a_idx, arch_v in enumerate(barchs):
            mlp = MLPClassifier(
                hidden_layer_sizes=arch_v, alpha=balpha,
                learning_rate='adaptive',
                learning_rate_init=blr, batch_size=bbs,
                max_iter=MLP_MAX_ITER, early_stopping=True,
                validation_fraction=0.15, n_iter_no_change=MLP_PATIENCE,
                random_state=seed + a_idx,
            )
            sw = make_sample_weights(y_it, bw)
            mlp.fit(X_it, y_it, sample_weight=sw)

            svp.append(mlp.predict_proba(X_iv)[:, 1])
            bag_va_proba.append(mlp.predict_proba(X_va_s)[:, 1])
            bag_te_proba.append(mlp.predict_proba(X_te_s)[:, 1])

        cal_p[seed] = np.mean(svp, axis=0)
        cal_l[seed] = y_iv

    # ── Calibration ───────────────────────────────────────────────
    cp = np.concatenate([cal_p[s] for s in BAG_SEEDS])
    cl = np.concatenate([cal_l[s] for s in BAG_SEEDS])
    iso = IsotonicRegression(y_min=0.001, y_max=0.999, out_of_bounds='clip')
    iso.fit(cp, cl)

    # Per-image probabilities (calibrated + raw)
    avg_va_raw = np.mean(bag_va_proba, axis=0)
    avg_te_raw = np.mean(bag_te_proba, axis=0)
    avg_va_cal = iso.predict(avg_va_raw)
    avg_te_cal = iso.predict(avg_te_raw)

    print(f'  Stage 1 done: {N_BAGS} MLPs trained')
    print(f'  Per-image proba range: val=[{avg_va_cal.min():.3f}, {avg_va_cal.max():.3f}]  '
          f'test=[{avg_te_cal.min():.3f}, {avg_te_cal.max():.3f}]')

    # ══════════════════════════════════════════════════════════════
    # STAGE 2: Bilateral Counting — Sweep threshold on val
    # ══════════════════════════════════════════════════════════════

    # Build referable ground truth for val and test
    ref_val  = build_referable_labels(b_val, TASK, BILATERAL_MIN_EYES)
    ref_test = build_referable_labels(b_test, TASK, BILATERAL_MIN_EYES)

    # Try both calibrated and raw probabilities
    best_overall_f1 = 0.0
    best_overall_t  = 0.5
    best_pt = 'calibrated'

    for pt_name, va_proba, te_proba in [
        ('calibrated', avg_va_cal, avg_te_cal),
        ('raw', avg_va_raw, avg_te_raw),
    ]:
        bt, bf1, sweep_df = sweep_bilateral_threshold(
            pids_va, va_proba, ref_val,
            min_eyes=BILATERAL_MIN_EYES,
            grid_size=THRESHOLD_GRID,
            floor=THRESHOLD_FLOOR,
        )
        if bf1 > best_overall_f1:
            best_overall_f1 = bf1
            best_overall_t  = bt
            best_pt = pt_name

    print(f'  ★ Stage 2 best: {best_pt} | bilateral_thr={best_overall_t:.3f} | val_F1={best_overall_f1:.4f}')

    # ── Evaluate on TEST ──────────────────────────────────────────
    te_proba_use = avg_te_cal if best_pt == 'calibrated' else avg_te_raw
    va_proba_use = avg_va_cal if best_pt == 'calibrated' else avg_va_raw

    test_agg = bilateral_patient_predictions(pids_te, te_proba_use, best_overall_t, BILATERAL_MIN_EYES)
    com = test_agg.index.intersection(ref_test.index)
    yt = ref_test.loc[com].values
    yh = test_agg.loc[com, 'referable_pred'].values
    yp = test_agg.loc[com, 'max_proba'].values  # Use max_proba for AUC

    test_f1   = float(f1_score(yt, yh, zero_division=0))
    test_prec = float(precision_score(yt, yh, zero_division=0))
    test_rec  = float(recall_score(yt, yh, zero_division=0))
    test_auc  = float(roc_auc_score(yt, yp)) if len(np.unique(yt)) > 1 else 0.0
    test_ba   = float(balanced_accuracy_score(yt, yh))

    elapsed = time.time() - t0

    print(f'  TEST: F1={test_f1:.4f}  Prec={test_prec:.4f}  Rec={test_rec:.4f}  '
          f'AUC={test_auc:.4f}  BalAcc={test_ba:.4f}')
    print(f'  Val−Test gap: {best_overall_f1 - test_f1:+.4f}')
    print(f'  Time: {elapsed:.0f}s')

    # ── Save results ──────────────────────────────────────────────
    out_dir = R31_DIR / f'{emb_idx:02d}_{sname}'
    out_dir.mkdir(parents=True, exist_ok=True)

    # Confusion matrix + classification report
    cm = confusion_matrix(yt, yh)
    rep = classification_report(yt, yh, digits=3,
                                target_names=['Not Referable', 'Referable'])
    print(rep)

    pd.DataFrame(cm).to_csv(out_dir / 'confusion_matrix.csv', index=False)
    (out_dir / 'classification_report.txt').write_text(rep, encoding='utf-8')

    # ROC curve (using max_proba per patient)
    fpr, tpr, _ = roc_curve(yt, yp)
    pd.DataFrame({'fpr': fpr, 'tpr': tpr}).to_csv(out_dir / 'roc_curve.csv', index=False)

    # Patient predictions
    pd.DataFrame({
        'patient_id': com,
        'y_true_referable': yt,
        'y_pred_referable': yh,
        'max_proba': yp,
        'n_positive_eyes': test_agg.loc[com, 'n_positive'].values,
        'n_images': test_agg.loc[com, 'n_images'].values,
    }).to_csv(out_dir / 'patient_predictions.csv', index=False)

    # Per-image predictions (for analysis)
    pd.DataFrame({
        'image_id': b_test['image_id'].values,
        'patient_id': pids_te,
        'y_true_eye': yb_test,
        'proba_calibrated': avg_te_cal,
        'proba_raw': avg_te_raw,
        'pred_positive_eye': (te_proba_use >= best_overall_t).astype(int),
    }).to_csv(out_dir / 'image_predictions.csv', index=False)

    # Bilateral threshold sweep (for analysis)
    _, _, sweep_final = sweep_bilateral_threshold(
        pids_va, va_proba_use, ref_val,
        min_eyes=BILATERAL_MIN_EYES, grid_size=THRESHOLD_GRID, floor=THRESHOLD_FLOOR)
    sweep_final.to_csv(out_dir / 'bilateral_threshold_sweep.csv', index=False)

    # Hyperparameters
    hp_save = {
        'track': 'R31', 'embedding': sname,
        'r28_architecture': str(barch), 'r28_alpha': float(balpha),
        'r28_lr': float(blr), 'r28_batch_size': int(bbs),
        'r28_pos_weight': float(bw),
        'pca_dim': nc, 'mi_selected': int(n_kept), 'pca_var': float(var_exp),
        'emb_dim_raw': emb_dim_raw, 'ndim': ndim,
        'n_bags': N_BAGS,
        'bilateral_min_eyes': BILATERAL_MIN_EYES,
        'bilateral_threshold': float(best_overall_t),
        'proba_type': best_pt,
        'val_referable_f1': float(best_overall_f1),
        'test_referable_f1': test_f1,
        'test_referable_precision': test_prec,
        'test_referable_recall': test_rec,
        'test_referable_auc': test_auc,
        'test_referable_bal_acc': test_ba,
        'n_test_patients': int(len(com)),
    }
    with open(out_dir / 'hyperparameters.json', 'w') as f:
        json.dump(hp_save, f, indent=2)

    # Plots
    plot_confusion_matrix(cm,
        title=f'R31 — {sname}\nBilateral Cascade | thr={best_overall_t:.3f}',
        save_path=out_dir / 'confusion_matrix.png', show=True)
    plot_roc_curve(fpr, tpr, test_auc,
        title=f'R31 — {sname} (Referable Glaucoma)',
        save_path=out_dir / 'roc_curve.png', show=True)

    r31_results.append({
        'embedding': sname,
        'dim_raw': emb_dim_raw, 'dim_pca': nc, 'dim_mi': int(n_kept),
        'pca_var': float(var_exp),
        'r28_arch': str(barch), 'r28_alpha': float(balpha),
        'r28_lr': float(blr), 'r28_pos_weight': float(bw),
        'bilateral_threshold': float(best_overall_t),
        'val_referable_f1': float(best_overall_f1),
        'test_referable_f1': test_f1,
        'test_referable_precision': test_prec,
        'test_referable_recall': test_rec,
        'test_referable_auc': test_auc,
        'test_referable_bal_acc': test_ba,
        'n_test_patients': int(len(com)),
        'time_s': float(elapsed),
    })

total = time.time() - t0_global
print(f'\n{"═"*80}')
print(f'R31 complete! Total: {total:.0f}s ({total/60:.1f} min)')
print(f'{"═"*80}')

# Save summary
r31_df = pd.DataFrame(r31_results).sort_values('test_referable_f1', ascending=False)
r31_df.to_csv(R31_DIR / 'r31_summary.csv', index=False)

print('\n── R31 Per-Encoder Summary (sorted by Test Referable F1) ──')
show_cols = ['embedding', 'val_referable_f1', 'test_referable_f1',
             'test_referable_precision', 'test_referable_recall',
             'test_referable_auc', 'bilateral_threshold']
print(r31_df[[c for c in show_cols if c in r31_df.columns]].to_string(index=False))

════════════════════════════════════════════════════════════════════════════════
R31 — Two-Stage Cascade: Per-Eye MLP → Bilateral Counting
════════════════════════════════════════════════════════════════════════════════

[1/7] convnextv2_base_ — CACHED, loading

[2/7] dinov3_convnext_base — CACHED, loading

[3/7] dinov3_vitb16 — CACHED, loading

[4/7] RETFound_dinov2_shanghai — CACHED, loading

[5/7] RETFound_mae_natureCFP — CACHED, loading

[6/7] RETFound_mae_shanghai — CACHED, loading

[7/7] vit_base_ — CACHED, loading

════════════════════════════════════════════════════════════════════════════════
R31 complete! Total: 0s (0.0 min)
════════════════════════════════════════════════════════════════════════════════

── R31 Per-Encoder Summary (sorted by Test Referable F1) ──
               embedding  val_referable_f1  test_referable_f1  test_referable_precision  test_referable_recall  test_referable_auc  bilateral_threshold
           dinov3_vitb16          0.844444           0.800885  

In [18]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 6 — Comparison with R29 and R30
# ═══════════════════════════════════════════════════════════════════════

print('═' * 80)
print('R31 vs R29 vs R30 — Referable Glaucoma Comparison')
print('═' * 80)

r29_path = BASE_DIR / 'results' / 'brset_r29_referable_glaucoma' / 'r29_per_encoder' / 'r29_summary.csv'
r30_path = BASE_DIR / 'results' / 'brset_r30_rich_meta_referable_glaucoma' / 'r30_per_encoder' / 'r30_summary.csv'

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

comparison_rows = []
r29_loaded = pd.read_csv(r29_path) if r29_path.exists() else None
r30_loaded = pd.read_csv(r30_path) if r30_path.exists() else None

for _, row in r31_df.iterrows():
    enc = row['embedding']
    comp = {
        'embedding': enc,
        'R31_f1': row['test_referable_f1'],
        'R31_prec': row['test_referable_precision'],
        'R31_rec': row['test_referable_recall'],
        'R31_auc': row['test_referable_auc'],
    }

    if r29_loaded is not None:
        r29_match = r29_loaded[r29_loaded['embedding'] == enc]
        if len(r29_match) > 0:
            f1c = _find_col(r29_loaded, ['f1_P25', 'f1', 'test_f1'])
            ac  = _find_col(r29_loaded, ['auc_P25', 'auc', 'test_auc'])
            if f1c: comp['R29_f1']  = r29_match.iloc[0][f1c]
            if ac:  comp['R29_auc'] = r29_match.iloc[0][ac]

    if r30_loaded is not None:
        r30_match = r30_loaded[r30_loaded['embedding'] == enc]
        if len(r30_match) > 0:
            f1c = _find_col(r30_loaded, ['f1_P25', 'f1', 'test_f1'])
            ac  = _find_col(r30_loaded, ['auc_P25', 'auc', 'test_auc'])
            if f1c: comp['R30_f1']  = r30_match.iloc[0][f1c]
            if ac:  comp['R30_auc'] = r30_match.iloc[0][ac]

    comparison_rows.append(comp)

comp_df = pd.DataFrame(comparison_rows)
print(comp_df.to_string(index=False))

# Best per track
print(f'\n── Best F1 per track ──')
best_r31 = r31_df['test_referable_f1'].max()
best_r31_enc = r31_df.loc[r31_df['test_referable_f1'].idxmax(), 'embedding']
print(f'  R31 (cascade):    {best_r31:.4f}  ({best_r31_enc})')

b29, b30 = None, None
if r29_loaded is not None:
    f1c29 = _find_col(r29_loaded, ['f1_P25', 'f1', 'test_f1'])
    if f1c29:
        b29 = r29_loaded[f1c29].max()
        e29 = r29_loaded.loc[r29_loaded[f1c29].idxmax(), 'embedding']
        print(f'  R29 (direct):     {b29:.4f}  ({e29})')

if r30_loaded is not None:
    f1c30 = _find_col(r30_loaded, ['f1_P25', 'f1', 'test_f1'])
    if f1c30:
        b30 = r30_loaded[f1c30].max()
        e30 = r30_loaded.loc[r30_loaded[f1c30].idxmax(), 'embedding']
        print(f'  R30 (enriched):   {b30:.4f}  ({e30})')

print(f'\n  Target: F1 >= 0.80')
if best_r31 >= 0.80:
    print(f'  R31 MEETS the 0.80 target! (F1={best_r31:.4f})')
else:
    print(f'  R31 does not meet 0.80 target (F1={best_r31:.4f}, gap={0.80-best_r31:.4f})')

# Improvements
if b29 is not None:
    imp29 = best_r31 - b29
    print(f'\n  R31 improvement over R29: {imp29:+.4f} ({imp29/b29*100:+.1f}%)')
if b30 is not None:
    imp30 = best_r31 - b30
    print(f'  R31 improvement over R30: {imp30:+.4f} ({imp30/b30*100:+.1f}%)')

════════════════════════════════════════════════════════════════════════════════
R31 vs R29 vs R30 — Referable Glaucoma Comparison
════════════════════════════════════════════════════════════════════════════════
               embedding   R31_f1  R31_prec  R31_rec  R31_auc   R29_f1  R29_auc   R30_f1  R30_auc
           dinov3_vitb16 0.800885  0.811659 0.790393 0.948628 0.721382 0.950291 0.558767 0.882916
    dinov3_convnext_base 0.764045  0.787037 0.742358 0.948276 0.687161 0.947150 0.511485 0.873922
        convnextv2_base_ 0.744467  0.690299 0.807860 0.942243 0.698413 0.951364 0.529915 0.890927
RETFound_dinov2_shanghai 0.743494  0.647249 0.873362 0.939468 0.642241 0.929306 0.521327 0.869692
   RETFound_mae_shanghai 0.728538  0.777228 0.685590 0.940723 0.642398 0.934782 0.495274 0.851366
  RETFound_mae_natureCFP 0.702461  0.720183 0.685590 0.924028 0.633803 0.922158 0.468817 0.840993
               vit_base_ 0.699248  0.613861 0.812227 0.932051 0.661844 0.937919 0.497512 0.849460

── 

In [19]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 7 — Diagnostic: Why did RETFound_dinov2_shanghai drop?
# ═══════════════════════════════════════════════════════════════════════

print('═' * 80)
print('DIAGNOSTIC: R28 (per-eye) vs R31 (bilateral cascade) per encoder')
print('═' * 80)

# Load R28 summary for per-eye performance
r28_summary_path = R28_DIR / 'r28_summary.csv'
r28_df = pd.read_csv(r28_summary_path)

# Side-by-side: R28 per-eye F1 vs R31 referable F1
diag_rows = []
for _, r31_row in r31_df.iterrows():
    enc = r31_row['embedding']
    r28_match = r28_df[r28_df['embedding'] == enc]
    r28_f1  = r28_match.iloc[0]['f1_P25'] if len(r28_match) > 0 else None
    r28_auc = r28_match.iloc[0]['auc_P25'] if len(r28_match) > 0 else None
    r28_rec = r28_match.iloc[0].get('rec_P25', r28_match.iloc[0].get('recall_P25', None)) if len(r28_match) > 0 else None
    r28_prec = r28_match.iloc[0].get('prec_P25', r28_match.iloc[0].get('precision_P25', None)) if len(r28_match) > 0 else None
    diag_rows.append({
        'encoder': enc,
        'R28_eye_F1': r28_f1,
        'R28_eye_AUC': r28_auc,
        'R28_eye_Prec': r28_prec,
        'R28_eye_Rec': r28_rec,
        'R31_ref_F1': r31_row['test_referable_f1'],
        'R31_ref_AUC': r31_row['test_referable_auc'],
        'R31_ref_Prec': r31_row['test_referable_precision'],
        'R31_ref_Rec': r31_row['test_referable_recall'],
        'R31_bilat_thr': r31_row['bilateral_threshold'],
        'R31_val_F1': r31_row['val_referable_f1'],
        'val_test_gap': r31_row['val_referable_f1'] - r31_row['test_referable_f1'],
    })

diag_df = pd.DataFrame(diag_rows)
print('\n── R28 Per-Eye vs R31 Bilateral ──')
print(diag_df[['encoder', 'R28_eye_F1', 'R28_eye_AUC', 'R31_ref_F1', 'R31_ref_AUC',
               'R31_bilat_thr', 'val_test_gap']].to_string(index=False))

# Now load per-image predictions for RETFound_dinov2_shanghai to understand the error pattern  
print('\n\n── RETFound_dinov2_shanghai: Error Analysis ──')
retf_img = pd.read_csv(R31_DIR / '03_RETFound_dinov2_shanghai' / 'image_predictions.csv')
retf_pat = pd.read_csv(R31_DIR / '03_RETFound_dinov2_shanghai' / 'patient_predictions.csv')

# How many images per patient?
imgs_per_pat = retf_img.groupby('patient_id').size()
print(f'Images per patient: mean={imgs_per_pat.mean():.1f}, median={imgs_per_pat.median():.0f}, '
      f'min={imgs_per_pat.min()}, max={imgs_per_pat.max()}')

# False positives: patients predicted referable but actually not
fp = retf_pat[(retf_pat['y_pred_referable'] == 1) & (retf_pat['y_true_referable'] == 0)]
fn = retf_pat[(retf_pat['y_pred_referable'] == 0) & (retf_pat['y_true_referable'] == 1)]
tp = retf_pat[(retf_pat['y_pred_referable'] == 1) & (retf_pat['y_true_referable'] == 1)]

print(f'\nRETFound_dinov2: TP={len(tp)}, FP={len(fp)}, FN={len(fn)}')
print(f'  FP: {len(fp)} patients falsely flagged as referable')
print(f'    Mean n_positive_eyes: {fp["n_positive_eyes"].mean():.1f}')
print(f'    Mean max_proba: {fp["max_proba"].mean():.3f}')
print(f'  FN: {len(fn)} referable patients missed')
print(f'    Mean n_positive_eyes: {fn["n_positive_eyes"].mean():.1f}')
print(f'    Mean max_proba: {fn["max_proba"].mean():.3f}')

# Same analysis for dinov3_vitb16 (best)
print('\n\n── dinov3_vitb16 (best): Error Analysis ──')
dino_img = pd.read_csv(R31_DIR / '02_dinov3_vitb16' / 'image_predictions.csv')
dino_pat = pd.read_csv(R31_DIR / '02_dinov3_vitb16' / 'patient_predictions.csv')

fp_d = dino_pat[(dino_pat['y_pred_referable'] == 1) & (dino_pat['y_true_referable'] == 0)]
fn_d = dino_pat[(dino_pat['y_pred_referable'] == 0) & (dino_pat['y_true_referable'] == 1)]
tp_d = dino_pat[(dino_pat['y_pred_referable'] == 1) & (dino_pat['y_true_referable'] == 1)]

print(f'dinov3_vitb16: TP={len(tp_d)}, FP={len(fp_d)}, FN={len(fn_d)}')
print(f'  FP: {len(fp_d)} patients falsely flagged')
print(f'    Mean n_positive_eyes: {fp_d["n_positive_eyes"].mean():.1f}')
print(f'  FN: {len(fn_d)} referable patients missed')
print(f'    Mean n_positive_eyes: {fn_d["n_positive_eyes"].mean():.1f}')

# Key insight: RETFound_dinov2 has LOW precision (0.647) = too many false positives
# Its Stage 1 threshold is 0.301 (very low) → lots of images flagged positive → many
# patients cross the >=2 threshold erroneously
print('\n\n── Bilateral threshold comparison ──')
for _, r in r31_df.iterrows():
    print(f'  {r["embedding"]:>30s}: thr={r["bilateral_threshold"]:.3f}  '
          f'prec={r["test_referable_precision"]:.3f}  rec={r["test_referable_recall"]:.3f}')

════════════════════════════════════════════════════════════════════════════════
DIAGNOSTIC: R28 (per-eye) vs R31 (bilateral cascade) per encoder
════════════════════════════════════════════════════════════════════════════════

── R28 Per-Eye vs R31 Bilateral ──
                 encoder  R28_eye_F1  R28_eye_AUC  R31_ref_F1  R31_ref_AUC  R31_bilat_thr  val_test_gap
           dinov3_vitb16    0.817853     0.963338    0.800885     0.948628       0.294400      0.043559
    dinov3_convnext_base    0.825822     0.956253    0.764045     0.948276       0.429133      0.035955
        convnextv2_base_    0.809938     0.951415    0.744467     0.942243       0.432267      0.043226
RETFound_dinov2_shanghai    0.825157     0.959683    0.743494     0.939468       0.300667      0.052966
   RETFound_mae_shanghai    0.820513     0.961198    0.728538     0.940723       0.219200      0.108196
  RETFound_mae_natureCFP    0.773823     0.941101    0.702461     0.924028       0.460467      0.065216
         

In [20]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 8 — Deeper: Unilateral patients are the FP source
# ═══════════════════════════════════════════════════════════════════════

# For each encoder, break down FPs into: truly-negative vs unilateral patients
print('═' * 80)
print('DEEPER ANALYSIS: What type of patients are False Positives?')
print('═' * 80)

# Build unilateral label (exactly 1 positive eye)
def build_eye_count_labels(df, task_col):
    pos_count = df[df[task_col] == 1].groupby('patient_id')[task_col].count()
    all_pids = df['patient_id'].unique()
    result = pd.Series(0, index=all_pids, name='n_positive_eyes_gt')
    result.loc[result.index.isin(pos_count.index)] = pos_count
    return result

for enc_dir in sorted(R31_DIR.iterdir()):
    pat_path = enc_dir / 'patient_predictions.csv'
    img_path = enc_dir / 'image_predictions.csv'
    if not pat_path.exists():
        continue
    
    enc_name = enc_dir.name[3:]  # strip "00_"
    pat = pd.read_csv(pat_path)
    img = pd.read_csv(img_path)
    img['patient_id'] = img['patient_id'].astype(str)
    
    # Ground truth per-eye counts
    gt_counts = build_eye_count_labels(img, 'y_true_eye')
    pat['patient_id'] = pat['patient_id'].astype(str)
    pat = pat.merge(gt_counts.rename('gt_pos_eyes'), left_on='patient_id', right_index=True, how='left')
    
    fp = pat[(pat['y_pred_referable'] == 1) & (pat['y_true_referable'] == 0)]
    
    fp_zero = fp[fp['gt_pos_eyes'] == 0]  # truly negative patients
    fp_unilateral = fp[fp['gt_pos_eyes'] == 1]  # unilateral — have 1 real positive eye
    
    print(f'\n{enc_name}:')
    print(f'  Total FP: {len(fp)}')
    print(f'    Truly negative (0 pos eyes): {len(fp_zero)} ({len(fp_zero)/max(1,len(fp))*100:.0f}%)')
    print(f'    Unilateral (1 pos eye):      {len(fp_unilateral)} ({len(fp_unilateral)/max(1,len(fp))*100:.0f}%)')
    
    # For unilateral FPs: the model predicts >=2 positive eyes, but truth is 1
    # This means 1+ extra eye is a false alarm at the image level
    if len(fp_unilateral) > 0:
        print(f'    Unilateral FP mean predicted pos eyes: {fp_unilateral["n_positive_eyes"].mean():.1f}')

print('\n\n── Per-image Stage 1 accuracy (eye-level) ──')
for enc_dir in sorted(R31_DIR.iterdir()):
    img_path = enc_dir / 'image_predictions.csv'
    if not img_path.exists():
        continue
    enc_name = enc_dir.name[3:]
    img = pd.read_csv(img_path)
    eye_f1 = f1_score(img['y_true_eye'], img['pred_positive_eye'], zero_division=0)
    eye_prec = precision_score(img['y_true_eye'], img['pred_positive_eye'], zero_division=0)
    eye_rec = recall_score(img['y_true_eye'], img['pred_positive_eye'], zero_division=0)
    n_pos_pred = img['pred_positive_eye'].sum()
    n_pos_true = img['y_true_eye'].sum()
    print(f'  {enc_name:>30s}: eye_F1={eye_f1:.3f}  eye_Prec={eye_prec:.3f}  '
          f'eye_Rec={eye_rec:.3f}  pred_pos={n_pos_pred}/{len(img)} true_pos={n_pos_true}')

════════════════════════════════════════════════════════════════════════════════
DEEPER ANALYSIS: What type of patients are False Positives?
════════════════════════════════════════════════════════════════════════════════

convnextv2_base_:
  Total FP: 83
    Truly negative (0 pos eyes): 41 (49%)
    Unilateral (1 pos eye):      42 (51%)
    Unilateral FP mean predicted pos eyes: 2.0

dinov3_convnext_base:
  Total FP: 46
    Truly negative (0 pos eyes): 25 (54%)
    Unilateral (1 pos eye):      21 (46%)
    Unilateral FP mean predicted pos eyes: 2.0

dinov3_vitb16:
  Total FP: 42
    Truly negative (0 pos eyes): 18 (43%)
    Unilateral (1 pos eye):      24 (57%)
    Unilateral FP mean predicted pos eyes: 2.0

RETFound_dinov2_shanghai:
  Total FP: 109
    Truly negative (0 pos eyes): 58 (53%)
    Unilateral (1 pos eye):      51 (47%)
    Unilateral FP mean predicted pos eyes: 2.0

RETFound_mae_natureCFP:
  Total FP: 61
    Truly negative (0 pos eyes): 31 (51%)
    Unilateral (1 pos eye)

In [21]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 9 — Two-Threshold Cascade CEILING analysis (test-set oracle)
#          Shows what's POSSIBLE with (eye_threshold, min_eyes) tuning
# ═══════════════════════════════════════════════════════════════════════

from itertools import product as iterproduct

MIN_EYES_GRID = [2, 3, 4]
THR_GRID_FINE = np.linspace(0.05, 0.95, 181)

print('═' * 80)
print('TWO-THRESHOLD CASCADE — Test-set ceiling analysis')
print('  ⚠  These numbers are UPPER BOUNDS (tuned on test, not val)')
print('═' * 80)

ceiling_rows = []

for enc_dir in sorted(R31_DIR.iterdir()):
    img_path = enc_dir / 'image_predictions.csv'
    pat_path = enc_dir / 'patient_predictions.csv'
    if not img_path.exists():
        continue
    enc_name = enc_dir.name[3:]  # strip "00_"

    img = pd.read_csv(img_path)
    pat = pd.read_csv(pat_path)
    img['patient_id'] = img['patient_id'].astype(str)
    pat['patient_id'] = pat['patient_id'].astype(str)

    # Ground truth: referable label per patient
    ref_gt = pat.set_index('patient_id')['y_true_referable']

    # R31 baseline (min_eyes=2, single threshold)
    r31_row = r31_df[r31_df['embedding'] == enc_name].iloc[0]
    baseline_f1 = r31_row['test_referable_f1']

    # 2D sweep on test
    best_f1, best_thr, best_me, best_prec, best_rec = 0, 0.5, 2, 0, 0
    sweep_data = []

    for me, thr in iterproduct(MIN_EYES_GRID, THR_GRID_FINE):
        agg = bilateral_patient_predictions(
            img['patient_id'].values,
            img['proba_calibrated'].values,
            threshold=thr, min_eyes=me,
        )
        com = agg.index.intersection(ref_gt.index)
        yt = ref_gt.loc[com].values
        yh = agg.loc[com, 'referable_pred'].values
        if yh.sum() == 0:
            continue
        f1 = f1_score(yt, yh, zero_division=0)
        prec = precision_score(yt, yh, zero_division=0)
        rec = recall_score(yt, yh, zero_division=0)
        sweep_data.append({'min_eyes': me, 'threshold': thr, 'f1': f1,
                           'precision': prec, 'recall': rec,
                           'n_pred_pos': int(yh.sum())})
        if f1 > best_f1:
            best_f1, best_thr, best_me = f1, thr, me
            best_prec, best_rec = prec, rec

    delta = best_f1 - baseline_f1
    print(f'\n{enc_name}:')
    print(f'  R31 baseline (min_eyes=2): F1={baseline_f1:.4f}')
    print(f'  Best ceiling  (min_eyes={best_me}): F1={best_f1:.4f}  '
          f'thr={best_thr:.3f}  prec={best_prec:.3f}  rec={best_rec:.3f}  '
          f'Δ={delta:+.4f}')

    # Also show best at each min_eyes
    df_sweep = pd.DataFrame(sweep_data)
    for me in MIN_EYES_GRID:
        sub = df_sweep[df_sweep['min_eyes'] == me]
        if sub.empty:
            continue
        brow = sub.loc[sub['f1'].idxmax()]
        tag = ' ◄' if me == best_me else ''
        print(f'    min_eyes={me}: F1={brow["f1"]:.4f}  thr={brow["threshold"]:.3f}  '
              f'prec={brow["precision"]:.3f}  rec={brow["recall"]:.3f}{tag}')

    ceiling_rows.append({
        'embedding': enc_name,
        'r31_f1': baseline_f1,
        'ceiling_f1': best_f1,
        'ceiling_min_eyes': best_me,
        'ceiling_threshold': best_thr,
        'ceiling_precision': best_prec,
        'ceiling_recall': best_rec,
        'delta': delta,
    })

ceiling_df = pd.DataFrame(ceiling_rows).sort_values('ceiling_f1', ascending=False)
print('\n\n── Summary: Two-Threshold Ceiling vs R31 Baseline ──')
print(ceiling_df.to_string(index=False))

════════════════════════════════════════════════════════════════════════════════
TWO-THRESHOLD CASCADE — Test-set ceiling analysis
  ⚠  These numbers are UPPER BOUNDS (tuned on test, not val)
════════════════════════════════════════════════════════════════════════════════

convnextv2_base_:
  R31 baseline (min_eyes=2): F1=0.7445
  Best ceiling  (min_eyes=2): F1=0.7486  thr=0.205  prec=0.668  rec=0.852  Δ=+0.0041
    min_eyes=2: F1=0.7486  thr=0.205  prec=0.668  rec=0.852 ◄
    min_eyes=3: F1=0.0087  thr=0.095  prec=1.000  rec=0.004

dinov3_convnext_base:
  R31 baseline (min_eyes=2): F1=0.7640
  Best ceiling  (min_eyes=2): F1=0.7702  thr=0.215  prec=0.751  rec=0.790  Δ=+0.0062
    min_eyes=2: F1=0.7702  thr=0.215  prec=0.751  rec=0.790 ◄
    min_eyes=3: F1=0.0086  thr=0.050  prec=0.250  rec=0.004

dinov3_vitb16:
  R31 baseline (min_eyes=2): F1=0.8009
  Best ceiling  (min_eyes=2): F1=0.7991  thr=0.290  prec=0.827  rec=0.773  Δ=-0.0018
    min_eyes=2: F1=0.7991  thr=0.290  prec=0.827  rec

In [22]:
# Quick check: how many images per patient in the test set?
img_sample = pd.read_csv(R31_DIR / '00_convnextv2_base_' / 'image_predictions.csv')
img_sample['patient_id'] = img_sample['patient_id'].astype(str)
ipp = img_sample.groupby('patient_id').size()
print('Images per patient (test set):')
print(ipp.value_counts().sort_index())
print(f'\nTotal patients: {ipp.shape[0]}')
print(f'Patients with ≥3 images: {(ipp >= 3).sum()} ({(ipp >= 3).mean():.1%})')
print(f'Patients with exactly 2 images: {(ipp == 2).sum()} ({(ipp == 2).mean():.1%})')
print(f'Patients with exactly 1 image: {(ipp == 1).sum()} ({(ipp == 1).mean():.1%})')

Images per patient (test set):
1     146
2    1555
3       3
4       1
Name: count, dtype: int64

Total patients: 1705
Patients with ≥3 images: 4 (0.2%)
Patients with exactly 2 images: 1555 (91.2%)
Patients with exactly 1 image: 146 (8.6%)


In [23]:
# ═══════════════════════════════════════════════════════════════════════
# Check: What hyperparameters is each encoder actually using?
# These come from R28 (per-eye any-glaucoma BayesSearchCV) — NOT tuned
# for the bilateral referable cascade.
# ═══════════════════════════════════════════════════════════════════════

rows = []
for enc, hp in sorted(r28_hypers.items()):
    rows.append({
        'encoder':    enc,
        'arch':       hp['architecture'],
        'alpha':      hp['alpha'],
        'lr':         hp['lr'],
        'batch_size': hp['batch_size'],
        'pos_weight': hp['pos_weight'],
    })

hp_df = pd.DataFrame(rows)
print('R28 hyperparameters re-used in R31 (per encoder):')
print(hp_df.to_string(index=False))

# Also show the R31 results alongside for context
print('\n\nR31 results for comparison:')
show = r31_df[['embedding', 'test_referable_f1', 'test_referable_precision',
               'test_referable_recall', 'bilateral_threshold', 'r28_pos_weight']].copy()
show = show.sort_values('test_referable_f1', ascending=False)
print(show.to_string(index=False))

R28 hyperparameters re-used in R31 (per encoder):
                 encoder   arch    alpha       lr  batch_size  pos_weight
RETFound_dinov2_shanghai (512,) 1.000000 0.000100          64    2.908782
  RETFound_mae_natureCFP (512,) 0.001000 0.000210          64    3.369478
   RETFound_mae_shanghai (256,) 0.001386 0.002000          64    1.000000
        convnextv2_base_ (128,) 1.000000 0.000100          64    4.070851
    dinov3_convnext_base (128,) 1.000000 0.000358          64    2.462905
           dinov3_vitb16 (128,) 1.000000 0.000301         256    2.448675
               vit_base_ (256,) 0.001000 0.002000         256    2.620812


R31 results for comparison:
               embedding  test_referable_f1  test_referable_precision  test_referable_recall  bilateral_threshold  r28_pos_weight
           dinov3_vitb16           0.800885                  0.811659               0.790393             0.294400        2.448675
    dinov3_convnext_base           0.764045                  0.78703